In [1]:
import torch
import torch.nn as nn
import numpy as np
import tensorflow as tf
print("PyTorch:", torch.__version__)
print("TensorFlow:", tf.__version__)
print("All ready!")

PyTorch: 2.12.1+cpu
TensorFlow: 2.21.0
All ready!


In [2]:
# Recreate the CNN architecture
class ECG_CNN(nn.Module):
    def __init__(self, num_classes=4):
        super(ECG_CNN, self).__init__()
        self.conv1 = nn.Conv1d(1, 32, kernel_size=5, padding=2)
        self.conv2 = nn.Conv1d(32, 64, kernel_size=5, padding=2)
        self.conv3 = nn.Conv1d(64, 128, kernel_size=3, padding=1)
        self.pool = nn.MaxPool1d(2)
        self.bn1 = nn.BatchNorm1d(32)
        self.bn2 = nn.BatchNorm1d(64)
        self.bn3 = nn.BatchNorm1d(128)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.3)
        self.fc1 = nn.Linear(128 * 22, 256)
        self.fc2 = nn.Linear(256, 64)
        self.fc3 = nn.Linear(64, num_classes)
    
    def forward(self, x):
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.pool(x)
        x = self.relu(self.bn2(self.conv2(x)))
        x = self.pool(x)
        x = self.relu(self.bn3(self.conv3(x)))
        x = self.pool(x)
        x = x.view(x.size(0), -1)
        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.relu(self.fc2(x))
        x = self.dropout(x)
        x = self.fc3(x)
        return x

# Load saved model
model = ECG_CNN(num_classes=4)
model.load_state_dict(torch.load(
    r'C:\Users\vaish\Desktop\ml-journey\week4\ecg_cnn_model.pth',
    map_location='cpu'
))
model.eval()
print("Model loaded successfully!")

# Test with dummy input
dummy = torch.randn(1, 1, 180)
with torch.no_grad():
    output = model(dummy)
print("Output shape:", output.shape)
print("Model ready for conversion!")

Model loaded successfully!
Output shape: torch.Size([1, 4])
Model ready for conversion!


In [3]:
import subprocess
import os

# Step 1 — Export PyTorch model to ONNX
dummy_input = torch.randn(1, 1, 180)
onnx_path = r'C:\Users\vaish\Desktop\ml-journey\week4\ecg_cnn.onnx'

torch.onnx.export(
    model,
    dummy_input,
    onnx_path,
    export_params=True,
    opset_version=11,
    input_names=['ecg_input'],
    output_names=['class_output'],
    dynamic_axes={
        'ecg_input': {0: 'batch_size'},
        'class_output': {0: 'batch_size'}
    }
)

print("✓ Step 1: PyTorch → ONNX done!")
print(f"  Saved to: {onnx_path}")
print(f"  File size: {os.path.getsize(onnx_path)/1024:.1f} KB")

C:\Users\vaish\AppData\Local\Temp\ipykernel_28164\385089731.py:8: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W0623 18:02:04.953000 28164 site-packages\torch\onnx\_internal\exporter\_compat.py:133] Setting ONNX exporter to use operator set version 18 because the requested opset_version 11 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features


[torch.onnx] Obtain model graph for `ECG_CNN([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `ECG_CNN([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


C:\Users\vaish\AppData\Local\Programs\Python\Python313\Lib\copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
The model version conversion is not supported by the onnxscript version converter and fallback is enabled. The model will be converted using the onnx C API (target version: 11).
Failed to convert the model to the target version 11 using the ONNX C API. The model was not modified
Traceback (most recent call last):
  File "C:\Users\vaish\AppData\Local\Programs\Python\Python313\Lib\site-packages\onnxscript\version_converter\__init__.py", line 137, in call
    converted_proto = _c_api_utils.call_onnx_api(
        func=_partial_convert_version, model=model
    )
  File "C:\Users\vaish\AppData\Local\Programs\Python\Python313\Lib\site-packages\onnxscript\version_converter\_c_api_utils.py", line 65, in call_onnx_api
    result = func(proto)
  File "C:\Users\v

[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
✓ Step 1: PyTorch → ONNX done!
  Saved to: C:\Users\vaish\Desktop\ml-journey\week4\ecg_cnn.onnx
  File size: 19.9 KB


In [4]:
import tensorflow as tf
import numpy as np

# Step 1 — Create equivalent TensorFlow model
def create_tf_model():
    inputs = tf.keras.Input(shape=(180, 1))
    
    x = tf.keras.layers.Conv1D(32, 5, padding='same', activation='relu')(inputs)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.MaxPooling1D(2)(x)
    
    x = tf.keras.layers.Conv1D(64, 5, padding='same', activation='relu')(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.MaxPooling1D(2)(x)
    
    x = tf.keras.layers.Conv1D(128, 3, padding='same', activation='relu')(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.MaxPooling1D(2)(x)
    
    x = tf.keras.layers.Flatten()(x)
    x = tf.keras.layers.Dense(256, activation='relu')(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    x = tf.keras.layers.Dense(64, activation='relu')(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    outputs = tf.keras.layers.Dense(4, activation='softmax')(x)
    
    model_tf = tf.keras.Model(inputs, outputs)
    return model_tf

tf_model = create_tf_model()
tf_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
tf_model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)             │ (None, 180, 1)              │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv1d (Conv1D)                      │ (None, 180, 32)             │             192 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization                  │ (None, 180, 32)             │             128 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling1d (MaxPooling1D)         │ (None, 90, 32)              │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv1d_1 (Conv1D)                    │ (None, 90, 64)              │          10,304 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_1                │ (None, 90, 64)              │             256 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling1d_1 (MaxPooling1D)       │ (None, 45, 64)              │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv1d_2 (Conv1D)                    │ (None, 45, 128)             │          24,704 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_2                │ (None, 45, 128)             │             512 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling1d_2 (MaxPooling1D)       │ (None, 22, 128)             │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten (Flatten)                    │ (None, 2816)                │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 256)                 │         721,152 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 256)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 64)                  │          16,448 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_1 (Dropout)                  │ (None, 64)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_2 (Dense)                      │ (None, 4)                   │             260 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 773,956 (2.95 MB)

 Trainable params: 773,508 (2.95 MB)

 Non-trainable params: 448 (1.75 KB)

In [5]:
# Prepare data for TensorFlow
# TF expects (batch, length, channels) — opposite of PyTorch

# Load the beats data again
import wfdb
from collections import Counter
from sklearn.utils import resample
from sklearn.model_selection import train_test_split

records = ['100', '101', '103', '105', '106', '108', '109', '111', '112', '113']

def extract_beats_multirecord(record_ids, window_size=180):
    all_beats = []
    all_labels = []
    valid_symbols = {'N': 0, 'A': 1, 'V': 2, 'L': 3}
    
    for rec_id in record_ids:
        try:
            record = wfdb.rdrecord(rec_id, pn_dir='mitdb')
            annotation = wfdb.rdann(rec_id, 'atr', pn_dir='mitdb')
            ecg_signal = record.p_signal[:, 0]
            for i, sample in enumerate(annotation.sample):
                symbol = annotation.symbol[i]
                if symbol not in valid_symbols:
                    continue
                start = sample - window_size // 2
                end = sample + window_size // 2
                if start < 0 or end > len(ecg_signal):
                    continue
                beat = ecg_signal[start:end]
                min_val, max_val = beat.min(), beat.max()
                if max_val - min_val != 0:
                    beat = (beat - min_val) / (max_val - min_val)
                all_beats.append(beat)
                all_labels.append(valid_symbols[symbol])
        except:
            pass
    return np.array(all_beats), np.array(all_labels)

print("Loading data...")
beats, labels = extract_beats_multirecord(records)

# Balance dataset
target_count = 500
balanced_beats = []
balanced_labels = []
for label in range(4):
    idx = np.where(labels == label)[0]
    b = beats[idx]
    l = labels[idx]
    b, l = resample(b, l, n_samples=target_count,
                    replace=len(b) < target_count, random_state=42)
    balanced_beats.append(b)
    balanced_labels.append(l)

X = np.vstack(balanced_beats)
y = np.hstack(balanced_labels)

# Reshape for TensorFlow — (samples, length, channels)
X = X.reshape(-1, 180, 1)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print("Data ready!")

Loading data...
X_train shape: (1600, 180, 1)
X_test shape: (400, 180, 1)
Data ready!
